# Exploration de SIRENE (INSEE) : enrichissement par SIRET

**Objectif :** savoir ce que SIRENE peut ajouter à chaque opérateur bio, et comment l'ingérer sans saturer le disque.

SIRENE est le répertoire officiel des entreprises (SIREN) et de leurs établissements (SIRET).
On utilise le fichier **StockEtablissement** : une ligne par établissement, ~2,9 Go compressé.

**Source :** `data/raw/sirene/2026-11/StockEtablissement_echantillon_5000.csv`,
les **5 000 premières lignes** du fichier de novembre 2026, lues sans télécharger le fichier complet.

In [1]:
from pathlib import Path

import pandas as pd

RACINE = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "docker-compose.yml").exists())
RAW = RACINE / "data/raw"

sirene = pd.read_csv(RAW / "sirene/2026-11/StockEtablissement_echantillon_5000.csv",
                     dtype=str, keep_default_na=False)
annuaire = pd.read_csv(RAW / "agence_bio/2026-11-05/operateursbio.csv",
                       sep=";", dtype=str, keep_default_na=False)
print(sirene.shape)

(5000, 54)


## 1. Attention : cet échantillon n'est pas représentatif

Le fichier est **trié par SIREN**. Les 5 000 premières lignes sont donc des entreprises aux SIREN les plus petits,
c'est-à-dire parmi les plus anciennes. Toute statistique calculée ici serait **biaisée** : on ne s'en sert que
pour comprendre la **structure** (colonnes, codes, formats), pas pour mesurer des proportions.

C'est un **biais d'échantillonnage** : un échantillon pris « au début du fichier » n'est pas un échantillon au hasard.

In [2]:
print("SIREN de", sirene["siren"].min(), "à", sirene["siren"].max())
print("Part d'établissements fermés dans l'échantillon :",
      f"{(sirene['etatAdministratifEtablissement'] == 'F').mean():.0%}  ← biaisé, pas une vérité nationale")

SIREN de 000325175 à 006670947
Part d'établissements fermés dans l'échantillon : 95%  ← biaisé, pas une vérité nationale


## 2. Les 54 colonnes, regroupées par thème

On ne gardera que les colonnes utiles au cas d'usage (principe de **minimisation** du RGPD : ne collecter que le nécessaire).

In [3]:
THEMES = {
    "Identifiants": ["siren", "nic", "siret"],
    "Diffusion (RGPD)": ["statutDiffusionEtablissement"],
    "Vie de l'établissement": ["dateCreationEtablissement", "etatAdministratifEtablissement", "dateDebut",
                               "etablissementSiege", "nombrePeriodesEtablissement", "dateDernierTraitementEtablissement"],
    "Activité": ["activitePrincipaleEtablissement", "nomenclatureActivitePrincipaleEtablissement",
                 "activitePrincipaleNAF25Etablissement", "activitePrincipaleRegistreMetiersEtablissement"],
    "Taille": ["trancheEffectifsEtablissement", "anneeEffectifsEtablissement", "caractereEmployeurEtablissement"],
    "Localisation": ["codePostalEtablissement", "codeCommuneEtablissement", "libelleCommuneEtablissement",
                     "codePaysEtrangerEtablissement", "coordonneeLambertAbscisseEtablissement",
                     "coordonneeLambertOrdonneeEtablissement"],
}
classees = {c for cols in THEMES.values() for c in cols}
autres = [c for c in sirene.columns if c not in classees]
for theme, cols in THEMES.items():
    print(f"{theme:24} {cols}")
print(f"{'Non retenues':24} {len(autres)} colonnes (adresse détaillée, seconde adresse, enseignes…)")

Identifiants             ['siren', 'nic', 'siret']
Diffusion (RGPD)         ['statutDiffusionEtablissement']
Vie de l'établissement   ['dateCreationEtablissement', 'etatAdministratifEtablissement', 'dateDebut', 'etablissementSiege', 'nombrePeriodesEtablissement', 'dateDernierTraitementEtablissement']
Activité                 ['activitePrincipaleEtablissement', 'nomenclatureActivitePrincipaleEtablissement', 'activitePrincipaleNAF25Etablissement', 'activitePrincipaleRegistreMetiersEtablissement']
Taille                   ['trancheEffectifsEtablissement', 'anneeEffectifsEtablissement', 'caractereEmployeurEtablissement']
Localisation             ['codePostalEtablissement', 'codeCommuneEtablissement', 'libelleCommuneEtablissement', 'codePaysEtrangerEtablissement', 'coordonneeLambertAbscisseEtablissement', 'coordonneeLambertOrdonneeEtablissement']
Non retenues             31 colonnes (adresse détaillée, seconde adresse, enseignes…)


## 3. Le statut de diffusion : SIRENE applique déjà le RGPD

- `O` : diffusible ;
- `P` : **partiellement diffusible**. La personne (souvent un entrepreneur individuel) s'est opposée à la diffusion
  de ses données personnelles. Les champs concernés sont remplacés par `[ND]` (non diffusible).

In [4]:
print(sirene["statutDiffusionEtablissement"].value_counts())
masquees = (sirene[sirene["statutDiffusionEtablissement"] == "P"] == "[ND]").sum()
masquees[masquees > 0].rename("valeurs [ND]")

statutDiffusionEtablissement
O    4947
P      53
Name: count, dtype: int64


complementAdresseEtablissement                    53
numeroVoieEtablissement                           53
indiceRepetitionEtablissement                     53
dernierNumeroVoieEtablissement                    53
indiceRepetitionDernierNumeroVoieEtablissement    53
typeVoieEtablissement                             53
libelleVoieEtablissement                          53
codePostalEtablissement                           53
distributionSpecialeEtablissement                 53
codeCedexEtablissement                            53
libelleCedexEtablissement                         53
identifiantAdresseEtablissement                   53
coordonneeLambertAbscisseEtablissement            53
coordonneeLambertOrdonneeEtablissement            53
complementAdresse2Etablissement                   53
numeroVoie2Etablissement                          53
indiceRepetition2Etablissement                    53
typeVoie2Etablissement                            53
libelleVoie2Etablissement                     

Le **code commune reste diffusé**, même quand l'adresse est masquée : c'est exactement ce dont on a besoin.
En revanche, il ne faut **jamais** chercher à reconstituer une adresse masquée : la personne s'y est opposée.
Et `[ND]` doit être converti en valeur vide en silver, sinon il serait pris pour une vraie valeur.

## 4. L'activité : le code NAF

Le code **NAF** (nomenclature d'activités française) décrit l'activité principale. Exemples utiles pour la filière bio :
`01.xx` (agriculture), `10.xx` (industries alimentaires), `46.xx` (commerce de gros), `47.xx` (commerce de détail).

La nomenclature a changé plusieurs fois : la colonne `nomenclatureActivitePrincipaleEtablissement` dit **laquelle** est utilisée.
Un même code n'a pas le même sens d'une nomenclature à l'autre.

In [5]:
print(sirene["nomenclatureActivitePrincipaleEtablissement"].value_counts())
actifs = sirene[sirene["etatAdministratifEtablissement"] == "A"]
print("\nNomenclature des établissements actifs :", actifs["nomenclatureActivitePrincipaleEtablissement"].value_counts().to_dict())

nomenclatureActivitePrincipaleEtablissement
NAP        3154
NAF1993    1033
NAFRev2     622
NAFRev1     167
             24
Name: count, dtype: int64

Nomenclature des établissements actifs : {'NAFRev2': 255}


Les établissements **actifs** sont tous en **NAF rév. 2** : les anciennes nomenclatures ne concernent que des établissements fermés depuis longtemps.
Une nouvelle colonne, `activitePrincipaleNAF25Etablissement`, prépare le passage à la **NAF 2025**.
Règle : on garde le code NAF rév. 2 et sa nomenclature, et on prévoit la colonne NAF 2025 pour la suite.

In [6]:
naf25 = sirene["activitePrincipaleNAF25Etablissement"]
print("Lignes avec un code NAF 2025 :", (naf25 != "").sum(), "| exemples :", naf25[naf25 != ""].head(3).tolist())

Lignes avec un code NAF 2025 : 267 | exemples : ['32.12Y', '46.64J', '10.81Y']


## 5. La taille : tranche d'effectifs

L'effectif n'est pas donné en nombre exact mais en **tranche codée** (`NN` = pas de salarié, `01` = 1 ou 2, `02` = 3 à 5, … `12` = 20 à 49…).
Il faudra une table de correspondance (code → libellé, borne basse, borne haute) en silver.

In [7]:
sirene["trancheEffectifsEtablissement"].value_counts().head(10)

trancheEffectifsEtablissement
NN    4913
01      16
12      16
02      14
11      12
21      12
03       9
22       4
00       3
31       1
Name: count, dtype: int64

## 6. La localisation : ce que SIRENE apporte en plus de l'annuaire

L'annuaire ne donne qu'un **code postal**, qui couvre souvent plusieurs communes. SIRENE donne :
- le **code commune INSEE** exact de l'établissement : il lève l'ambiguïté du code postal ;
- des **coordonnées Lambert 93** (projection officielle française, en mètres), utilisables pour une carte fine.

In [8]:
print("Code commune renseigné (actifs)  :", f"{(actifs['codeCommuneEtablissement'] != '').mean():.0%}")
print("Coordonnées renseignées (actifs)  :",
      f"{(~actifs['coordonneeLambertAbscisseEtablissement'].isin(['', '[ND]'])).mean():.0%}")

Code commune renseigné (actifs)  : 100%
Coordonnées renseignées (actifs)  : 62%


## 7. Test de jointure avec l'annuaire

Même sur cet échantillon de 5 000 lignes, quelques SIRET de l'annuaire sont présents. On compare ce que disent les deux sources.

In [9]:
jointure = annuaire.merge(sirene, left_on="SIRET", right_on="siret")
jointure[["SIRET", "CODE POSTAL SIEGE SOCIAL", "codePostalEtablissement", "codeCommuneEtablissement",
          "etatAdministratifEtablissement", "activitePrincipaleEtablissement", "trancheEffectifsEtablissement"]]

,SIRET,CODE POSTAL SIEGE SOCIAL,codePostalEtablissement,codeCommuneEtablissement,etatAdministratifEtablissement,activitePrincipaleEtablissement,trancheEffectifsEtablissement
0,00638004200033,12700,12700,12052,A,10.86Z,31
1,00648020600017,44560,44530,44068,A,11.03Z,12
2,00645004300028,04700,04700,04143,A,10.39B,12
3,00565003100015,04240,04240,04008,A,10.13A,03


**Les deux sources peuvent se contredire** : ici, un code postal diffère (`44560` dans l'annuaire, `44530` dans SIRENE).
Explications possibles : l'annuaire donne le code postal du **siège social**, alors que le SIRET peut désigner un autre établissement ;
ou l'une des deux sources n'est pas à jour.

C'est une question de **gouvernance** : quelle source fait foi pour quelle information ?
- **Localisation de l'établissement** : SIRENE (registre officiel, code commune exact) ;
- **Activités bio, certificateur, date d'engagement** : l'annuaire de l'Agence Bio ;
- **Les écarts** sont gardés et comptés dans les contrôles qualité, sans être corrigés à la main.

## 8. Comment ingérer SIRENE sans saturer le disque

| Approche | Volume | Verdict |
|---|---|---|
| Télécharger, décompresser, tout charger | ~2,9 Go compressé, plusieurs Go décompressé, des dizaines de millions de lignes (établissements fermés compris) | Non : inutile, on n'a besoin que de ~87 000 SIRET |
| Lire le ZIP **en flux** et ne garder que les SIRET de l'annuaire | ~25 Mo gardés | **Oui** : c'est la méthode déjà utilisée pour Agreste |

Chaque mois, le job télécharge le fichier dans un dossier temporaire, le lit en flux, garde les lignes dont le SIRET est
dans l'annuaire (jointure par une table de recherche), écrit le résultat en bronze, puis **supprime** le fichier complet.

## 9. Synthèse : règles pour la couche silver

| Sujet | Règle |
|---|---|
| Colonnes | Ne garder que : identifiants, diffusion, état, dates, activité (NAF rév. 2 + nomenclature + NAF 2025), tranche d'effectifs, employeur, code commune, coordonnées. **Minimisation RGPD** |
| `[ND]` | Converti en valeur vide. Ne jamais reconstituer une donnée masquée |
| Clé | `siret` (14 caractères, texte). Jointure avec l'annuaire sur le SIRET nettoyé (chiffres seuls, Luhn valide) |
| État | `A` (actif) / `F` (fermé), avec `dateDebut` = date du dernier changement : utile pour l'axe 2 (fermetures) |
| Tranche d'effectifs | Table de correspondance code → libellé et bornes |
| Localisation | Le code commune SIRENE **prime** sur la déduction depuis le code postal. Les écarts sont comptés en contrôle qualité |
| Fréquence | Mensuelle (le stock est publié en début de mois) |

### Pour Talend

- `tFileFetch` (ou `tHttpRequest`) pour télécharger, puis `tFileUnarchive` dans un dossier temporaire.
- `tFileInputDelimited` (séparateur `,`, UTF-8, en-tête), puis `tMap` avec les SIRET de l'annuaire en **lookup**
  et le réglage « Inner Join » : seules les lignes trouvées passent.
- `tFileDelete` pour supprimer le fichier complet une fois l'extraction faite (règle de rétention du cadrage).
- À vérifier au moment du job : la mémoire. Un lookup de 87 000 SIRET tient facilement en mémoire, contrairement aux dizaines de millions de lignes du flux principal, qui doivent rester **en flux**.